In [1]:
#pip install -r requirements.txt

In [2]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
apiwrapper = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wikipedia_tool = WikipediaQueryRun(api_wrapper = apiwrapper)
wikipedia_tool.name 


'wikipedia'

In [3]:
wikipedia_tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en"
)
vectordb = FAISS.from_documents(documents,embeddings)
retriever = vectordb.as_retriever()
retriever




USER_AGENT environment variable not set, consider setting it to identify your requests.
C:\Users\Len\AppData\Local\Temp\ipykernel_25716\4014223785.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000012F00A1FC50>)

In [5]:
from langchain.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever,
    name="langsmith_search",
    description ="USE THIS TOOL whenever the user asks about LangChain tracing, evaluation, observability, or debugging. This tool contains the official LangSmith technical documentation."
)


In [6]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun
arxivapiwrapper = ArxivAPIWrapper(top_k_results=3,
    doc_content_chars=500, 
    load_max_docs=3,
    )
arxiv_tool= ArxivQueryRun(api_wrapper =arxivapiwrapper)
arxiv_tool.name



'arxiv'

In [7]:
tools = [wikipedia_tool,arxiv_tool,retriever_tool]

In [ ]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [9]:
# pip uninstall langchain langchain-core langchain-community langchain-groq -y


In [10]:
# ! pip install langchain==0.2.16 \
#     langchain-core==0.2.41 \
#             langchain-community==0.2.16 \
#             langchain-groq


In [11]:
from langchain import hub
prompt = hub.pull("hwchase17/react")
prompt

C:\Users\Len\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [17]:

from langchain.agents import AgentExecutor, create_react_agent
from langchain_groq import ChatGroq
from langchain.memory import ConversationSummaryBufferMemory

memory = ConversationSummaryBufferMemory(
    llm =llm,
    memory_key="chat_history",
    return_messages=True
)


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    streaming=False
)

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, #to see process
    max_iterations=10,
    max_execution_time=60,
    early_stopping_method="force",# 'force' simply stops, 'generate' tries one last time to summarize what it found
    handle_parsing_errors=True,
    memory=memory
)


In [13]:
print(agent_executor.invoke({"input":"You are a travel researcher. Find 5 beautiful uninhabited places. As soon as you have 5 examples from your tools, stop searching and provide them as your Final Answer immediately. Do not try to find every single place in the world"})["output"])



> Entering new AgentExecutor chain...
Thought: To find 5 beautiful uninhabited places, I should start by searching for general information on uninhabited locations around the world. This can include islands, forests, mountains, or other areas that are not populated by humans. I can use Wikipedia to search for a list of uninhabited places or articles about remote and beautiful locations.

Action: wikipedia
Action Input: "uninhabited islands"Page: Uninhabited island
Summary: An uninhabited island, desert island, or deserted island, is an island, islet or atoll which lacks permanent human population. Uninhabited islands are often depicted Thought: The Wikipedia search provided a general overview of uninhabited islands, but it didn't give me specific examples of beautiful uninhabited places. I'll try to search for a list of uninhabited islands to see if I can find some examples.

Action: wikipedia
Action Input: "list of uninhabited islands"Page: Uninhabited island
Summary: An uninhabited

In [18]:
agent_executor.invoke({"input":"explain me about human"})
agent_executor.invoke({"input":"why they have two legs and not four legs"})
agent_executor.invoke({"input":"how can i them as a robot"})





> Entering new AgentExecutor chain...
To answer a question about humans, I should first consider what aspect of humans the user is interested in learning about. This could include biology, psychology, sociology, or other fields of study. Given the broad nature of the question, I'll start by searching for general information about humans.

Action: wikipedia
Action Input: humanPage: Human
Summary: Humans (Homo sapiens, meaning "thinking man" or "wise man") are the most common and widespread species of primate, and the only extant species of the genus Homo. They are great apQuestion: explain me about human
Thought: To answer a question about humans, I should first consider what aspect of humans the user is interested in learning about. This could include biology, psychology, sociology, or other fields of study. Given the broad nature of the question, I'll start by searching for general information about humans.

Action: wikipedia
Action Input: humanPage: Human
Summary: Humans (Homo sapi

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

C:\Users\Len\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Len\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\Len\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(



> Finished chain.


> Entering new AgentExecutor chain...
Thought: The question seems to be related to the evolution or biology of a specific species, likely humans. To answer this question, I need to find information about the evolutionary history of humans and why they developed two legs instead of four.

Action: wikipedia
Action Input: human evolutionPage: Human evolution
Summary: Homo sapiens is a distinct species of the hominid family of primates, which also includes all the great apes. Over their evolutionary history, humans gradually developedQuestion: why they have two legs and not four legs
Thought: The question seems to be related to the evolution or biology of a specific species, likely humans. To answer this question, I need to find information about the evolutionary history of humans and why they developed two legs instead of four.

Action: wikipedia
Action Input: human evolutionPage: Human evolution
Summary: Homo sapiens is a distinct species of the hominid family of pri

{'input': 'how can i them as a robot',
 'chat_history': [HumanMessage(content='explain me about human'),
  AIMessage(content='Humans (Homo sapiens, meaning "thinking man" or "wise man") are the most common and widespread species of primate, and the only extant species of the genus Homo. They are great apes, characterized by their upright posture, binocular vision, and ability to use tools. Humans are a highly social species, with complex cultures, languages, and societies. They have a wide range of physical characteristics, including a diverse range of skin colors, hair textures, and body shapes. Humans are also capable of complex thought, emotion, and behavior, and have developed a wide range of technologies, arts, and sciences.'),
  HumanMessage(content='why they have two legs and not four legs'),
  AIMessage(content='Humans evolved to have two legs instead of four due to the advantages of bipedalism, which include the ability to see over obstacles, cover long distances at a relative

In [ ]:
# from flask import Flask, request
# from twilio.twiml.messaging_response import MessagingResponse
# import pandas as pd
# import os

# app = Flask(__name__)

# BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# # ---------- LOAD DATA ----------
# training = pd.read_csv(os.path.join(BASE_DIR, "Data", "Training.csv"))

# symptom_description = {}
# with open(os.path.join(BASE_DIR, "MasterData", "symptom_description.csv")) as f:
#     for line in f:
#         parts = line.strip().split(",")
#         if len(parts) >= 2:
#             symptom_description[parts[0].lower()] = parts[1]

# precaution = {}
# with open(os.path.join(BASE_DIR, "MasterData", "symptom_precaution.csv")) as f:
#     for line in f:
#         parts = line.strip().split(",")
#         if len(parts) >= 5:
#             precaution[parts[0].lower()] = parts[1:5]

# SYMPTOMS = [c.lower() for c in training.columns[:-1]]

# # ---------- WHATSAPP WEBHOOK ----------
# @app.route("/whatsapp", methods=["POST"])
# def whatsapp():
#     incoming = request.values.get("Body", "").lower().strip()
#     resp = MessagingResponse()
#     msg = resp.message()

#     # split symptoms
#     user_symptoms = incoming.replace(",", " ").split()
#     matched = [s for s in user_symptoms if s in SYMPTOMS]

#     if len(matched) == 0:
#         msg.body(
#             "❗ I couldn't detect valid symptoms.\n\n"
#             "Try:\n"
#             "* fever cough\n"
#             "* headache vomiting\n"
#             "* chest_pain"
#         )
#         return str(resp)

#     if len(matched) == 1:
#         msg.body("⚠️ Single-symptom prediction is not reliable.\nPlease send multiple symptoms.")
#         return str(resp)

#     # filter dataset
#     filtered = training.copy()
#     for s in matched:
#         filtered = filtered[filtered[s] == 1]

#     if filtered.empty:
#         msg.body("❗ No disease match found. Please try different symptom combinations.")
#         return str(resp)

#     disease = filtered["prognosis"].mode()[0].lower()

#     reply = f"🩺 Possible Disease: {disease.title()}\n\n"

#     if disease in symptom_description:
#         reply += f"📖 Description:\n{symptom_description[disease]}\n\n"

#     if disease in precaution:
#         reply += "✅ Precautions:\n"
#         for i, p in enumerate(precaution[disease], 1):
#             reply += f"{i}. {p}\n"

#     reply += "\n⚠️ Disclaimer: This is not a medical diagnosis. Please consult a doctor."

#     msg.body(reply)
#     return str(resp)

# # ---------- RUN ----------
# if __name__ == "__main__":
#     app.run(port=5000)